In [4]:
import customtkinter as ctk
from tkinter import filedialog, messagebox
from PIL import Image
import cv2
import numpy as np
import pandas as pd
import os
import threading

ctk.set_appearance_mode("Dark")
ctk.set_default_color_theme("blue")

PIKSEL_ESIK = 50
DERS_SAYISI = 8
ISIM_DOLULUK_ESIK = 100

class UltraModernYoklama(ctk.CTk):
    def __init__(self):
        super().__init__()
        self.title("Akıllı Yoklama Sistemi")
        self.geometry("1250x850")
        self.grid_columnconfigure(1, weight=1)
        self.grid_rowconfigure(0, weight=1)
        
        self.dosya_yolu = None
        self.sinif_listesi_df = None

        
        self.sidebar_frame = ctk.CTkFrame(self, width=280, corner_radius=0)
        self.sidebar_frame.grid(row=0, column=0, sticky="nsew")
        self.sidebar_frame.grid_rowconfigure(6, weight=1)

        self.logo_label = ctk.CTkLabel(self.sidebar_frame, text="AKILLI YOKLAMA\nSİSTEMİ", 
                                     font=ctk.CTkFont(size=24, weight="bold"))
        self.logo_label.grid(row=0, column=0, padx=20, pady=(40, 20))

        self.btn_liste = ctk.CTkButton(self.sidebar_frame, text="📋 Sınıf Listesi Yükle (Excel)", command=self.liste_yukle,
                                     height=40, fg_color="#E67E22", hover_color="#D35400", font=ctk.CTkFont(size=13, weight="bold"))
        self.btn_liste.grid(row=1, column=0, padx=20, pady=(10, 10), sticky="ew")

        self.btn_sec = ctk.CTkButton(self.sidebar_frame, text="📷 Fotoğraf Yükle", command=self.resim_sec,
                                     height=40, font=ctk.CTkFont(size=13, weight="bold"))
        self.btn_sec.grid(row=2, column=0, padx=20, pady=(10, 10), sticky="ew")

        self.btn_analiz = ctk.CTkButton(self.sidebar_frame, text="🚀 Analizi Başlat", command=self.analiz_baslat_thread,
                                        height=50, fg_color="transparent", border_width=2, 
                                        text_color=("gray10", "#DCE4EE"), state="disabled", font=ctk.CTkFont(size=14, weight="bold"))
        self.btn_analiz.grid(row=3, column=0, padx=20, pady=(20, 10), sticky="ew")

        self.lbl_liste_durum = ctk.CTkLabel(self.sidebar_frame, text="Liste: Yüklü Değil (Otomatik İsim)", font=ctk.CTkFont(size=12, slant="italic"), text_color="#E74C3C")
        self.lbl_liste_durum.grid(row=4, column=0, padx=20, pady=5)

        self.lbl_durum = ctk.CTkLabel(self.sidebar_frame, text="Sistem Bekliyor...", font=ctk.CTkFont(size=12))
        self.lbl_durum.grid(row=5, column=0, padx=20, pady=10)

        self.progress_bar = ctk.CTkProgressBar(self.sidebar_frame)
        self.progress_bar.grid(row=7, column=0, padx=20, pady=(10, 20), sticky="ew")
        self.progress_bar.set(0)

        self.main_frame = ctk.CTkFrame(self, corner_radius=20, fg_color=("gray95", "gray10"))
        self.main_frame.grid(row=0, column=1, padx=20, pady=20, sticky="nsew")
        self.main_frame.grid_columnconfigure(0, weight=1)
        self.main_frame.grid_rowconfigure(1, weight=1)

        self.lbl_info = ctk.CTkLabel(self.main_frame, text="Önce Sınıf Listesi (Opsiyonel), sonra Yoklama Kağıdı seçiniz.", 
                                     font=ctk.CTkFont(size=16), text_color="gray50")
        self.lbl_info.grid(row=0, column=0, pady=(20, 10))

        self.resim_kutusu = ctk.CTkLabel(self.main_frame, text="", corner_radius=10)
        self.resim_kutusu.grid(row=1, column=0, padx=20, pady=20)

    def liste_yukle(self):
        dosya = filedialog.askopenfilename(filetypes=[("Excel Dosyaları", "*.xlsx;*.xls")])
        if dosya:
            try:
                df = pd.read_excel(dosya)
                if len(df.columns) < 1:
                    raise ValueError("Excel dosyası boş veya formatı hatalı.")
                
                self.sinif_listesi_df = df
                dosya_adi = os.path.basename(dosya)
                self.lbl_liste_durum.configure(text=f"Liste: {dosya_adi} ✅", text_color="#2ECC71")
                messagebox.showinfo("Başarılı", f"Sınıf listesi yüklendi!\n{len(df)} öğrenci bulundu.")
            except Exception as e:
                messagebox.showerror("Hata", f"Excel okunurken hata oluştu:\n{e}")

    def resim_oku_tr(self, yol):
        try:
            with open(yol, "rb") as f:
                bytes = bytearray(f.read())
                numpyarray = np.asarray(bytes, dtype=np.uint8)
                img = cv2.imdecode(numpyarray, cv2.IMREAD_COLOR)
            return img
        except Exception as e:
            return None

    def resim_sec(self):
        filename = filedialog.askopenfilename(filetypes=[("Resimler", "*.png;*.jpg;*.jpeg")])
        if filename:
            self.dosya_yolu = filename
            img_cv = self.resim_oku_tr(filename)
            if img_cv is not None:
                self.resim_goster(None, opencv_img=img_cv)
                self.lbl_durum.configure(text="Resim Hazır ✅")
                self.lbl_info.configure(text=f"Seçilen: {os.path.basename(filename)}")
                self.btn_analiz.configure(state="normal", fg_color="#2CC985", text_color="white", border_width=0)
            else:
                messagebox.showerror("Hata", "Resim okunamadı!")

    def resim_goster(self, path, opencv_img=None):
        if opencv_img is None: img = Image.open(path)
        else: img = Image.fromarray(cv2.cvtColor(opencv_img, cv2.COLOR_BGR2RGB))
        ctk_img = ctk.CTkImage(light_image=img, dark_image=img, size=(800, 550))
        self.resim_kutusu.configure(image=ctk_img)

    def analiz_baslat_thread(self):
        threading.Thread(target=self.analizi_yap, daemon=True).start()

    def analizi_yap(self):
        if not self.dosya_yolu: return
        try:
            self.lbl_durum.configure(text="İşleniyor... ⏳")
            self.progress_bar.start()
            
            img_orj = self.resim_oku_tr(self.dosya_yolu)
            if img_orj is None: raise ValueError("Resim Hatası")

            h, w = img_orj.shape[:2]
            oran = 1000 / float(h)
            img = cv2.resize(img_orj, (int(w * oran), 1000))

            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            thresh = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 51, 10)
            
            h_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (40, 1))
            v_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, 40))
            grid_img = cv2.addWeighted(cv2.morphologyEx(thresh, cv2.MORPH_OPEN, h_kernel), 1, 
                                       cv2.morphologyEx(thresh, cv2.MORPH_OPEN, v_kernel), 1, 0.0)
            grid_img = cv2.dilate(grid_img, np.ones((3,3), np.uint8), iterations=1)

            contours, _ = cv2.findContours(grid_img, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
            kutucuklar = []
            for c in contours:
                x, y, w, h_rect = cv2.boundingRect(c)
                if w > 20 and h_rect > 15 and w < 800 and h_rect < 150:
                    kutucuklar.append((x, y, w, h_rect))
            kutucuklar.sort(key=lambda b: b[1])

            satirlar = []
            mevcut_satir = []
            ilk_y = kutucuklar[0][1]
            for kutu in kutucuklar:
                if abs(kutu[1] - ilk_y) < 20: mevcut_satir.append(kutu)
                else:
                    mevcut_satir.sort(key=lambda b: b[0])
                    satirlar.append(mevcut_satir)
                    mevcut_satir = [kutu]
                    ilk_y = kutu[1]
            satirlar.append(mevcut_satir)

            sonuc_img = img.copy()
            yoklama_verisi = []
            kernel_analiz = np.ones((2,2), np.uint8)
            baslik_gecildi = False

            for satir in satirlar:
                if len(satir) < DERS_SAYISI + 1: continue

                if not baslik_gecildi:
                    for kutu in satir:
                        cv2.rectangle(sonuc_img, (kutu[0], kutu[1]), (kutu[0]+kutu[2], kutu[1]+kutu[3]), (255, 0, 0), 2)
                    baslik_gecildi = True
                    continue

                ders_kutulari = satir[-DERS_SAYISI:]
                bilgi_kutulari = satir[:-DERS_SAYISI]

                if len(bilgi_kutulari) > 0:
                    isim_kutusu = bilgi_kutulari[-1]
                    roi_isim = thresh[isim_kutusu[1]+5:isim_kutusu[1]+isim_kutusu[3]-5, 
                                      isim_kutusu[0]+5:isim_kutusu[0]+isim_kutusu[2]-5]
                    if cv2.countNonZero(roi_isim) < ISIM_DOLULUK_ESIK:
                        continue 
    
                if self.sinif_listesi_df is not None:
                    if len(yoklama_verisi) >= len(self.sinif_listesi_df): break
                
                for kutu in bilgi_kutulari:
                    cv2.rectangle(sonuc_img, (kutu[0], kutu[1]), (kutu[0]+kutu[2], kutu[1]+kutu[3]), (255, 0, 0), 2)

                gecici_veri = []
                for kutu in ders_kutulari:
                    x, y, w_rect, h_rect = kutu
                    roi = thresh[y+5:y+h_rect-5, x+5:x+w_rect-5]
                    roi = cv2.dilate(roi, kernel_analiz, iterations=1)
                    durum = 1 if cv2.countNonZero(roi) > PIKSEL_ESIK else 0
                    renk = (0, 255, 0) if durum == 1 else (0, 0, 255)
                    cv2.rectangle(sonuc_img, (x, y), (x+w_rect, y+h_rect), renk, 2)
                    gecici_veri.append(durum)
                yoklama_verisi.append(gecici_veri)

            self.progress_bar.stop()
            self.progress_bar.set(1)
            self.resim_goster(None, opencv_img=sonuc_img)
            
            if len(yoklama_verisi) > 0:
                cols = [f"Ders {k+1}" for k in range(DERS_SAYISI)]
                df_yoklama = pd.DataFrame(yoklama_verisi, columns=cols)
                
                if self.sinif_listesi_df is not None:
                    df_liste = self.sinif_listesi_df.copy()
                    
                    min_len = min(len(df_liste), len(df_yoklama))
                    df_sonuc = pd.concat([df_liste.iloc[:min_len].reset_index(drop=True), 
                                          df_yoklama.iloc[:min_len].reset_index(drop=True)], axis=1)
                else:
                    otomatik_isimler = [{"Sıra": i+1, "Öğrenci": f"Öğrenci {i+1}"} for i in range(len(yoklama_verisi))]
                    df_liste = pd.DataFrame(otomatik_isimler)
                    df_sonuc = pd.concat([df_liste, df_yoklama], axis=1)

                df_sonuc["Toplam Gelinen Gün"] = df_sonuc[cols].sum(axis=1)
                
                kayit_adi = "Yoklama_Sonucu_Final.xlsx"
                df_sonuc.to_excel(kayit_adi, index=False)
                
                self.lbl_durum.configure(text="Kayıt Başarılı! 📊", text_color="#2CC985")
                messagebox.showinfo("Başarılı", f"İşlem Tamamlandı!\n\n📂 Kaydedilen Dosya:\n{kayit_adi}\n\n👥 Tespit Edilen Kişi: {len(yoklama_verisi)}")
            else:
                self.lbl_durum.configure(text="Hata: Satır Bulunamadı ❌", text_color="red")

        except Exception as e:
            self.progress_bar.stop()
            self.lbl_durum.configure(text="Hata Oluştu!", text_color="red")
            messagebox.showerror("Kritik Hata", str(e))

if __name__ == "__main__":
    app = UltraModernYoklama()
    app.mainloop()